In [1]:
!pip -q install azure-ai-documentintelligence azure-core
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 7.9 MB/s eta 0:00:00


In [2]:
"""
Batch OCR of all documents in an Azure Blob container
using Azure AI Document Intelligence (Prebuilt Read)
"""

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.storage.blob import ContainerClient
import numpy as np
import pandas as pd

# -------------------------
# Secrets (Kaggle)
# -------------------------
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

endpoint = user_secrets.get_secret("AZURE_DOCINT_ENDPOINT")
key = user_secrets.get_secret("AZURE_DOCINT_KEY")
container_sas_url = user_secrets.get_secret("AZURE_BLOB_CONTAINER_SAS_URL")

print("Endpoint loaded:", bool(endpoint))
print("Key loaded:", bool(key))
print("Container SAS loaded:", bool(container_sas_url))

# -------------------------
# Clients
# -------------------------
docint_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

container_client = ContainerClient.from_container_url(
    container_sas_url
)


import re

def extract_invoice_fields(text: str) -> dict:
    def find(pattern, cast=str):
        match = re.search(pattern, text)
        return cast(match.group(1)) if match else None

    return {
        "invoice_number": find(r"Invoice\s*#:\s*([A-Z0-9\-]+)"),
        "invoice_date": find(r"Date:\s*([0-9]{2}/[0-9]{2}/[0-9]{4})"),
        "vendor": find(r"Company:\s*([A-Za-z0-9 ]+)"),
        "total_due": find(
            r"Total Due:\s*\$?([0-9,]+\.\d{2})",
            lambda x: float(x.replace(",", ""))
        )
    }


# -------------------------
# Helpers
# -------------------------
def format_bounding_box(bounding_box):
    if not bounding_box:
        return "N/A"
    reshaped = np.array(bounding_box).reshape(-1, 2)
    return ", ".join(f"[{x}, {y}]" for x, y in reshaped)

# -------------------------
# Analyze all blobs
# -------------------------
CONFIDENCE_THRESHOLD = 0.9
invoices = []

def analyze_all_documents():
    for blob in container_client.list_blobs():

        #Only process document files (PDFs and images). Skip everything else:
        if not blob.name.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
            continue

        base_url, sas = container_sas_url.split("?", 1)
        blob_url = f"{base_url}/{blob.name}?{sas}"

        # Prints processing Document: CompanyName1.pdf
        print("\n" + "=" * 60)
        print(f"📄 Processing Document: {blob.name}")
        print("=" * 60)

        poller = docint_client.begin_analyze_document(
            "prebuilt-read",
            AnalyzeDocumentRequest(url_source=blob_url)
        )
        result = poller.result()

        # ---- Extract BUSINESS fields ----
        invoice = extract_invoice_fields(result.content)
        invoice["source_file"] = blob.name
        invoices.append(invoice)

        # ---- OPTIONAL: OCR diagnostics ----
        for page in result.pages:
            low_conf_words = [
                word for word in page.words
                if word.confidence is not None
                and word.confidence < CONFIDENCE_THRESHOLD
            ]

            if low_conf_words:
                print("\n⚠️ Low-confidence words")
                for word in low_conf_words:
                    print(
                        f"'{word.content}' "
                        f"{word.confidence:.2f} "
                        f"Box: {format_bounding_box(word.polygon)}"
                    )


def export_to_excel(data):
    df = pd.DataFrame(data)

    output_file = "invoice_summary.xlsx"
    df.to_excel(output_file, index=False)

    print(f"\n✅ Exported {len(df)} invoices to '{output_file}'")


# -------------------------
# Entry point
# -------------------------
if __name__ == "__main__":
    analyze_all_documents()
    export_to_excel(invoices)

Endpoint loaded: True
Key loaded: True
Container SAS loaded: True

📄 Processing Document: Amazon/Amazon_Invoice_1.pdf

📄 Processing Document: Amazon/Amazon_Invoice_2.pdf

📄 Processing Document: Amazon/Amazon_Invoice_3.pdf

📄 Processing Document: CompanyAlpha.pdf

📄 Processing Document: CompanyBeta.pdf

📄 Processing Document: CompanyName1.pdf

📄 Processing Document: CompanyName2.pdf

📄 Processing Document: CompanyName3.pdf

📄 Processing Document: sample_invoice.pdf

⚠️ Low-confidence words
'Illinois' 0.89 Box: [4.1489, 1.882], [4.5522, 1.8833], [4.5449, 2.0479], [4.1404, 2.0451]

📄 Processing Document: sample_invoice2.pdf

⚠️ Low-confidence words
'#:' 0.77 Box: [1.4764, 1.2738], [1.5867, 1.2742], [1.5838, 1.4011], [1.4741, 1.4012]

📄 Processing Document: sandbox_invoice.pdf

✅ Exported 11 invoices to 'invoice_summary.xlsx'
